## Import Libraries and Load data

In [2]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV

In [3]:
DATA_PATH= './../data'

In [4]:
main_df = pd.read_csv(os.path.join(DATA_PATH,'CNA_SV.csv'))
main_df.head()

,Unnamed: 0,9-Sep_SV,A2BP1_SV,AASDH_SV,AATF_SV,AATK_SV,AATK-AS1_SV,ABCA1_SV,ABCA12_SV,ABCA13_SV,...,TSC2_CNA,TSHR_CNA,U2AF1_CNA,VHL_CNA,WT1_CNA,XPO1_CNA,ZRSR2_CNA,SEX,AGE_AT_SEQ_REPORT,CANCER_TYPE
0,GENIE-DFCI-000024-4434,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Female,87,Esophagogastric Cancer
1,GENIE-DFCI-000027-10441,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,Female,74,Soft Tissue Sarcoma
2,GENIE-DFCI-000029-10065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-1.0,-1.0,0.0,0.0,0.0,Male,63,Renal Cell Carcinoma
3,GENIE-DFCI-000029-526237,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-1.0,-1.0,-1.0,0.0,0.0,0.0,Male,65,Renal Cell Carcinoma
4,GENIE-DFCI-000035-11184,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,Female,67,CNS Cancer


## Data Preprocessing

In [ ]:
df_CNA_SV = main_df[main_df['CANCER_TYPE'] != 'Cancer of Unknown Primary']
test_df = main_df[main_df['CANCER_TYPE'] == 'Cancer of Unknown Primary']


# Filter based on cancer types with more than 1000 instances
cancer_count = df_CNA_SV['CANCER_TYPE'].value_counts()
selected_cancer_types = cancer_count[cancer_count.values > 1000].index.to_list()
df_CNA_SV = df_CNA_SV[df_CNA_SV['CANCER_TYPE'].isin(selected_cancer_types)].reset_index(drop=True)

# Drop columns with zero variation
zero_variation_col = []
for col in df_CNA_SV.columns:
    if len(df_CNA_SV[col].unique()) <= 1:
        zero_variation_col.append(col)


df = df_CNA_SV.copy()
df=df.drop(['Unnamed: 0']+zero_variation_col,axis=1).reset_index(drop=True)

In [ ]:
## Label encode cancer types
label_encoder_cancer = LabelEncoder()
df['CANCER_TYPE'] = label_encoder_cancer.fit_transform(df['CANCER_TYPE'])

In [ ]:
#Remove low correlated features
import pickle

with open(os.path.join(DATA_PATH,'specific_column_corr.pickle'), 'rb') as handle:
    specific_column_corr_loaded = pickle.load(handle)

unselected_features=[col for col in df.columns if col not in specific_column_corr_loaded.index.to_list()]

df=df.drop(unselected_features,axis=1).reset_index(drop=True)
df.shape

In [ ]:
X_test =test_df.drop(['Unnamed: 0','CANCER_TYPE'] + zero_variation_col+unselected_features,axis=1).reset_index(drop=True)
X_test.shape

In [ ]:
# Split the data into features and target
X = df.drop(columns=['CANCER_TYPE'])
y = df['CANCER_TYPE']

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
#Data transformation

label_encoder_sex = LabelEncoder()
X_train['SEX'] = label_encoder_sex.fit_transform(X_train[['SEX']])

scaler = StandardScaler()
X_train['AGE_AT_SEQ_REPORT'] = scaler.fit_transform(X_train[['AGE_AT_SEQ_REPORT']])

X_val['SEX'] = label_encoder_sex.transform(X_val['SEX'])
X_val['AGE_AT_SEQ_REPORT'] = scaler.transform(X_val[['AGE_AT_SEQ_REPORT']])


X_test['SEX'] = label_encoder_sex.transform(X_test['SEX'])
X_test['AGE_AT_SEQ_REPORT'] = scaler.transform(X_test[['AGE_AT_SEQ_REPORT']])

## Hyperparameter search with GridSearchCV

In [27]:
# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'scale_pos_weight': [1, 10, 25]  # Adjusting scale_pos_weight for imbalanced data
}

# Initialize the XGBoost model
xgb_model = xgb.XGBClassifier(random_state=42)

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)

# Fit GridSearchCV
grid_search.fit(X_train, y_train)

# Get the best model
best_xgb_model = grid_search.best_estimator_

# Make predictions with the best model
y_pred = best_xgb_model.predict(X_val)

# Evaluate the model
accuracy = accuracy_score(y_val, y_pred)
print(f'Best XGBoost Accuracy: {accuracy:.4f}')
print(classification_report(y_val, y_pred, target_names=label_encoder_cancer.classes_))

# Print the best parameters
print(f'Best parameters: {grid_search.best_params_}')


Fitting 3 folds for each of 729 candidates, totalling 2187 fits


/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain fea

[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=3, n_estimators=100, scale_pos_weight=1, subsample=0.8; total time=   2.7s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=3, n_estimators=100, scale_pos_weight=25, subsample=0.6; total time=   2.8s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=3, n_estimators=200, scale_pos_weight=10, subsample=0.6; total time=   5.0s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=3, n_estimators=200, scale_pos_weight=25, subsample=1.0; total time=   4.9s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=3, n_estimators=300, scale_pos_weight=10, subsample=0.8; total time=   7.4s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=5, n_estimators=100, scale_pos_weight=1, subsample=0.8; total time=   3.2s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=5, n_estimators=100, scale_pos_weight=10, subsample=0.6; total time=   3.1s
[CV] END colsample_bytree=0.6, learning_rat

/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:40:10] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:40:10] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:40:10] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:40:11] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)


[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=3, n_estimators=100, scale_pos_weight=1, subsample=1.0; total time=   2.7s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=3, n_estimators=100, scale_pos_weight=25, subsample=1.0; total time=   2.6s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=3, n_estimators=200, scale_pos_weight=10, subsample=0.6; total time=   5.4s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=3, n_estimators=300, scale_pos_weight=1, subsample=0.8; total time=   8.8s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=3, n_estimators=300, scale_pos_weight=25, subsample=0.6; total time=   7.4s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=5, n_estimators=100, scale_pos_weight=10, subsample=0.8; total time=   3.2s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=5, n_estimators=200, scale_pos_weight=1, subsample=0.8; total time=   6.3s
[CV] END colsample_bytree=0.6, learning_rate

...

[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=100, scale_pos_weight=10, subsample=1.0; total time=   2.7s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=100, scale_pos_weight=25, subsample=1.0; total time=   2.9s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=200, scale_pos_weight=10, subsample=0.8; total time=   5.3s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=300, scale_pos_weight=1, subsample=0.8; total time=   7.5s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=300, scale_pos_weight=25, subsample=0.6; total time=   7.6s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=5, n_estimators=100, scale_pos_weight=10, subsample=0.8; total time=   3.2s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=5, n_estimators=200, scale_pos_weight=1, subsample=0.6; total time=   5.8s
[CV] END colsample_bytree=0.8, learning_rate=0.1, 

/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:47:17] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:47:18] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:47:18] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:47:18] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:47:19] WARNING: /workspace/s

[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=200, scale_pos_weight=25, subsample=1.0; total time=   5.4s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=300, scale_pos_weight=10, subsample=0.8; total time=   7.4s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=5, n_estimators=100, scale_pos_weight=1, subsample=0.8; total time=   3.3s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=5, n_estimators=100, scale_pos_weight=10, subsample=0.6; total time=   3.2s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=5, n_estimators=100, scale_pos_weight=25, subsample=0.8; total time=   4.1s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=5, n_estimators=200, scale_pos_weight=10, subsample=1.0; total time=   6.7s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=5, n_estimators=300, scale_pos_weight=1, subsample=0.8; total time=   8.3s
[CV] END colsample_bytree=0.8, learning_rate=0.1, 

/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:47:33] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:47:33] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:47:33] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:47:37] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:47:37] WARNING: /workspace/s

Best XGBoost Accuracy: 0.6751
                            precision    recall  f1-score   support

             Breast Cancer       0.68      0.79      0.73       596
         Colorectal Cancer       0.68      0.52      0.59       350
    Esophagogastric Cancer       0.66      0.38      0.48       225
                    Glioma       0.88      0.69      0.77       351
Non-Small Cell Lung Cancer       0.56      0.79      0.66       823
            Ovarian Cancer       0.69      0.34      0.45       233
           Prostate Cancer       0.88      0.78      0.82       440
       Soft Tissue Sarcoma       0.64      0.61      0.63       281

                  accuracy                           0.68      3299
                 macro avg       0.71      0.61      0.64      3299
              weighted avg       0.69      0.68      0.67      3299

Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 300, 'scale_pos_weight': 1, 'subsample': 0.8}


#### Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 300, 'scale_pos_weight': 1, 'subsample': 0.8}

## XGBoost classifier

In [9]:
# Train an XGBoost classifier
xgb_model = xgb.XGBClassifier(colsample_bytree=0.8, learning_rate= 0.2, max_depth=3, n_estimators=300, scale_pos_weight=1, subsample= 0.8, random_state=42)
xgb_model.fit(X_train, y_train)

# Make predictions

y_pred = xgb_model.predict(X_train)
# Evaluate the model
train_accuracy = accuracy_score(y_train, y_pred)
print(f'XGBoost training Accuracy: {train_accuracy:.4f}')

y_pred = xgb_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_pred)
print(f'XGBoost validation Accuracy: {val_accuracy:.4f}')
print(classification_report(y_val, y_pred, target_names=label_encoder_cancer.classes_))

/home/afiai97/.local/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [22:36:26] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost training Accuracy: 0.7631
XGBoost validation Accuracy: 0.6751
                            precision    recall  f1-score   support

             Breast Cancer       0.68      0.79      0.73       596
         Colorectal Cancer       0.68      0.52      0.59       350
    Esophagogastric Cancer       0.66      0.38      0.48       225
                    Glioma       0.88      0.69      0.77       351
Non-Small Cell Lung Cancer       0.56      0.79      0.66       823
            Ovarian Cancer       0.69      0.34      0.45       233
           Prostate Cancer       0.88      0.78      0.82       440
       Soft Tissue Sarcoma       0.64      0.61      0.63       281

                  accuracy                           0.68      3299
                 macro avg       0.71      0.61      0.64      3299
              weighted avg       0.69      0.68      0.67      3299

